In [0]:
SELECT 
    po.full_name AS receptor_ofensivo,
    po.team AS equipo,
    pd.full_name AS esquinero_defensor,
    ROUND(AVG(f.separation_yards), 2) AS promedio_separacion_yardas,
    ROUND(MAX(f.separation_yards), 2) AS maxima_separacion_yardas,
    COUNT(DISTINCT f.play_sk) AS total_jugadas_analizadas
FROM dev_nfl_telemetry_kafka.dev_nfl_telemetry_kafka_gold.platinum_fact_separation f

-- 1. Unimos la dimensión del jugador OFENSIVO
JOIN dev_nfl_telemetry_kafka.dev_nfl_telemetry_kafka_gold.gold_dim_player po 
    ON f.offensive_player_sk = po.player_sk

-- 2. Unimos la dimensión del jugador DEFENSIVO (reutilizando la misma tabla)
JOIN dev_nfl_telemetry_kafka.dev_nfl_telemetry_kafka_gold.gold_dim_player pd 
    ON f.defensive_player_sk = pd.player_sk

-- 3. Unimos la dimensión de la JUGADA
JOIN dev_nfl_telemetry_kafka.dev_nfl_telemetry_kafka_gold.gold_dim_play pl 
    ON f.play_sk = pl.play_sk

-- Filtramos solo momentos de alta presión (3er Down)
WHERE pl.down = 3
  
-- Agrupamos por los jugadores involucrados
GROUP BY 
    po.full_name,
    po.team,
    pd.full_name

-- Filtramos para tener una muestra estadística válida (mínimo 3 jugadas enfrentándose)
HAVING total_jugadas_analizadas >= 3

-- Ordenamos para ver a los receptores más letales hasta arriba
ORDER BY 
    promedio_separacion_yardas DESC
LIMIT 10;